# ZS601: ten-view no-glass depth comparison
Method A: verified 1 cm point-cloud initialization, k=3 RMS × 0.5, nominal opacity 1, no training.
Reuses the authenticated v006 Colab runtime and pinned CUDA renderer. Exact input hashes are checked.
The companion CPU z-buffer and PNG backprojection evaluator run locally from the same ten cameras.
Saved depth: uint16 PNG, single-channel camera Z in millimetres, 0 invalid.


In [1]:
from pathlib import Path
import json,sys,subprocess,torch
ROOT=Path('/content/zs601-depth-methods-v007')
OLD=Path('/content/zs601-mesh-noglass-v006')
assert torch.cuda.is_available()
print(dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(),
           sigmoid20_float32=float(torch.sigmoid(torch.tensor(20.,device='cuda')).cpu())))
print(json.loads((ROOT/'run_spec.json').read_text()))


{'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA L4', 'sigmoid20_float32': 1.0}
{'run_id': 'zs601-depth-methods-v007', 'random_seed': 20260922, 'selection': '10 without replacement from fixed 200 virtual_near cameras; no quality-based selection', 'view_ids': [3202, 3340, 3376, 3388, 3409, 3481, 3505, 3520, 3571, 3583], 'views': 10, 'resolution': [640, 1108], 'input_cloud': 'D:\\codex\\blenderProject\\3dgsResult\\zs601-mesh-noglass-v006\\cloud_1cm\\points_mesh_1cm_noglass.ply', 'input_cloud_sha256': '68f26fca61b4c1424c1f45f2f4983c4c42cf389abc23d6e0afadd9d48819de5b', 'source_gaussian': 'D:\\codex\\blenderProject\\3dgsResult\\zs601-mesh-noglass-v006\\full\\point_cloud\\iteration_0\\point_cloud.ply', 'source_gaussian_sha256': 'c984cd38e3d31e34c1db4d4afb41196bc4980fe50f1ddf924a599568fa221411', 'source_run': 'zs601-mesh-noglass-v006', 'source_geometry': 'D:\\codex\\blenderProject\\scenes\\zs601-meetingroom\\synthetic-training-v001\\source\\geometry.npz', 'excluded_glass_faces': 'D:\\

In [2]:
subprocess.run([sys.executable,'-u',str(ROOT/'run_gaussian_depth.py'),
 '--package',str(OLD/'source/gaussian-splatting-lidar-init'),
 '--source-gaussian',str(OLD/'run/full/point_cloud/iteration_0/point_cloud.ply'),
 '--input-cloud',str(OLD/'input/points_mesh_1cm_noglass.ply'),
 '--spec',str(ROOT/'run_spec.json'),'--views',str(ROOT/'selected_views.json'),
 '--output',str(ROOT/'method_a_gaussian')],check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/zs601-depth-methods-v007/run_gaussian_depth.py', '--package', '/content/zs601-mesh-noglass-v006/source/gaussian-splatting-lidar-init', '--source-gaussian', '/content/zs601-mesh-noglass-v006/run/full/point_cloud/iteration_0/point_cloud.ply', '--input-cloud', '/content/zs601-mesh-noglass-v006/input/points_mesh_1cm_noglass.ply', '--spec', '/content/zs601-depth-methods-v007/run_spec.json', '--views', '/content/zs601-depth-methods-v007/selected_views.json', '--output', '/content/zs601-depth-methods-v007/method_a_gaussian'], returncode=0)

In [3]:
from PIL import Image
import numpy as np
out=ROOT/'method_a_gaussian'
views=json.loads((ROOT/'selected_views.json').read_text())
assert len(views)==10
for folder in ['images','alpha','masks','depth','depth_mask']:
    assert len(list((out/folder).glob('*.png')))==10
for v in views:
    d=np.array(Image.open(out/'depth'/v['name']))
    m=np.array(Image.open(out/'depth_mask'/v['name']))>0
    assert d.dtype==np.uint16 and np.array_equal(d>0,m)
    assert d.shape==(v['height'],v['width'])
print((out/'initialization.json').read_text())
print((out/'render_summary.json').read_text())
print('TEN_VIEW_GAUSSIAN_DEPTH_PASS_NO_TRAINING')


{
  "point_count": 7004696,
  "sh_degree": 0,
  "optimization_steps": 0,
  "source_gaussian_sha256": "c984cd38e3d31e34c1db4d4afb41196bc4980fe50f1ddf924a599568fa221411",
  "input_cloud_sha256": "68f26fca61b4c1424c1f45f2f4983c4c42cf389abc23d6e0afadd9d48819de5b",
  "gaussian_ply_sha256": "430011f5e22c20359ecdaa2aeec621439b7c3ea56ed1500ab9f1478264973c28",
  "changed_parameter": "opacity only",
  "knn_k": 3,
  "scale_factor": 0.5,
  "sigma_formula": "0.5 * sqrt(mean(d_nearest3 ** 2))",
  "nominal_opacity": 1.0,
  "actual_float32_opacity": {
    "min": 1.0,
    "median": 1.0,
    "max": 1.0
  },
  "opacity_logits": {
    "min": 20.0,
    "median": 20.0,
    "max": 20.0
  },
  "mathematical_sigmoid20": 0.9999999979388463,
  "kernel_per_fragment_alpha_cap": 0.99,
  "kernel_near_cutoff_m": 0.2,
  "scales_metres": {
    "min": 0.0006092521362006664,
    "median": 0.004407880827784538,
    "max": 0.009362607263028622
  },
  "ply_reload_exact": true,
  "xyz_equal_input": true,
  "source_other_fiel